# Delimiter-guided non-binary bottom-up oracle with Stanza trees

This notebook contains the cleaned pipeline for the delimiter-guided bottom-up system, using Stanza's constituency tree representation:

```python
from stanza.models.constituency.parse_tree import Tree
from stanza.models.constituency.tree_reader import read_trees
```

The pipeline:

1. read a PTB-style tree with `read_trees`,
2. extract the sentence,
3. derive the static bottom-up oracle,
4. apply the oracle to reconstruct a Stanza `Tree`,
5. check exact reconstruction against the original tree.

The transition inventory is:

\[
\textsc{Shift},\ \textsc{d-Shift},\ \textsc{Nt}(X),\ \textsc{u-Reduce},\ \textsc{Reduce},\ \textsc{d-Reduce},\ \textsc{Finish}.
\]

Unlike the previous arity-based non-binary bottom-up system, the phrase-level reduction action does not specify an arity. The span is recovered from the nearest delimiter. `FINISH` is an explicit termination action, used to verify that the buffer has been consumed and the stack contains exactly one completed tree.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable, List, Optional, Sequence, Tuple, Union

from stanza.models.constituency.parse_tree import Tree
from stanza.models.constituency.tree_reader import read_trees

OracleOp = Tuple[str, Optional[str]]

/Users/jungyeul/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jungyeul/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tree utilities

The notebook uses Stanza `Tree` objects throughout.  The small wrapper functions below isolate Stanza-specific tree access, so the oracle and applicator remain readable.

In [2]:
def parse_ptb(ptb_string: str) -> Tree:
    """Parse one PTB-style bracketed tree into a Stanza constituency Tree."""
    trees = list(read_trees(ptb_string, use_tqdm=False))
    if not trees:
        raise ValueError(f"No tree found in string: {ptb_string!r}")
    if len(trees) != 1:
        raise ValueError(f"Expected exactly one tree, found {len(trees)}.")
    return trees[0]


def label_of(tree: Tree) -> str:
    """Return the label stored on a Stanza Tree node."""
    return tree.label


def children_of(tree: Tree) -> List[Tree]:
    """Return the children of a Stanza Tree node as a list."""
    return list(tree.children or [])


def make_tree(label: str, children: Optional[Sequence[Tree]] = None) -> Tree:
    """Construct a Stanza Tree node.

    Stanza versions expose the same logical constructor, but this helper keeps
    construction localized in case a local environment uses keyword-only fields.
    """
    children = list(children or [])
    try:
        return Tree(label, children)
    except TypeError:
        return Tree(label=label, children=children)


def is_leaf(tree: Tree) -> bool:
    return len(children_of(tree)) == 0


def is_preterminal(tree: Tree) -> bool:
    children = children_of(tree)
    return len(children) == 1 and is_leaf(children[0])


def terminal_word(preterminal: Tree) -> str:
    """Return the terminal word dominated by a preterminal."""
    if not is_preterminal(preterminal):
        raise ValueError(f"Expected a preterminal, found: {tree_to_ptb(preterminal)}")
    return label_of(children_of(preterminal)[0])


def get_sentence(tree: Tree) -> List[str]:
    """Return the terminal yield of a Stanza Tree."""
    if is_leaf(tree):
        return [label_of(tree)]

    words: List[str] = []
    for child in children_of(tree):
        words.extend(get_sentence(child))
    return words


def tree_to_ptb(tree: Tree) -> str:
    """Return a single-line PTB string from a Stanza Tree."""
    if is_leaf(tree):
        return label_of(tree)
    return f"({label_of(tree)} {' '.join(tree_to_ptb(child) for child in children_of(tree))})"


def same_tree(left: Tree, right: Tree) -> bool:
    """Exact structural comparison through the canonical PTB string form."""
    return tree_to_ptb(left) == tree_to_ptb(right)

## Static oracle

The oracle traverses the gold tree bottom-up. A preterminal is built immediately after shifting its terminal, using `NT(POS)` followed by `U-REDUCE`.

A delimiter is introduced by `D-SHIFT` when the current terminal begins a local reduction domain that must be preserved until a higher constituent is complete. Phrase-level constituents are then built by `REDUCE` or `D-REDUCE`:

* `REDUCE` builds the constituent above the nearest delimiter and consumes that delimiter.
* `D-REDUCE` builds the constituent above the nearest delimiter but preserves the delimiter for a higher reduction.
* `FINISH` terminates the derivation after all input words have been consumed and exactly one completed tree remains on the stack.

In [3]:
@dataclass(frozen=True)
class DBoundary:
    """A delimiter introduced by D-SHIFT and consumed by its target node."""

    target_id: int


class TreeNavigator:
    """Small helper for parent and child-index lookup in a Stanza Tree."""

    def __init__(self, root: Tree):
        self.root = root
        self.parent_of: dict[int, Tree] = {}
        self.child_index_of: dict[int, int] = {}
        self._register(root)

    def _register(self, node: Tree) -> None:
        for index, child in enumerate(children_of(node)):
            self.parent_of[id(child)] = node
            self.child_index_of[id(child)] = index
            self._register(child)

    def highest_left_edge_phrasal_ancestor(self, preterminal: Tree) -> Optional[Tree]:
        """
        Return the highest phrasal ancestor reached by moving upward from a
        preterminal while the current node is the first child of its parent.

        The preterminal itself is not a target, because it is built by U-REDUCE.
        """
        current: Tree = preterminal
        target: Optional[Tree] = None

        while id(current) in self.parent_of:
            parent = self.parent_of[id(current)]
            child_index = self.child_index_of[id(current)]

            if child_index != 0:
                break

            if not is_preterminal(parent):
                target = parent

            current = parent

        return target


def get_oracle_bottomup(tree: Tree) -> List[OracleOp]:
    """
    Return the delimiter-guided non-binary bottom-up oracle for a gold tree.

    The oracle is deterministic for a fixed tree. It does not encode arity in
    phrase-level reduction actions.  The final action is always FINISH.
    """
    oracle: List[OracleOp] = []
    navigator = TreeNavigator(tree)

    previous_parent_id: Optional[int] = None
    terminal_index = 0

    def shift_word(word: str, parent: Optional[Tree], preterminal: Optional[Tree]) -> Optional[DBoundary]:
        nonlocal previous_parent_id, terminal_index

        current_parent_id = id(parent) if parent is not None else None
        is_first_terminal = terminal_index == 0
        parent_changed = previous_parent_id is not None and current_parent_id != previous_parent_id

        target = None
        if preterminal is not None:
            target = navigator.highest_left_edge_phrasal_ancestor(preterminal)

        use_d_shift = target is not None and (is_first_terminal or parent_changed)

        if use_d_shift:
            oracle.append(("D-SHIFT", word))
            boundary = DBoundary(target_id=id(target))
        else:
            oracle.append(("SHIFT", word))
            boundary = None

        previous_parent_id = current_parent_id
        terminal_index += 1
        return boundary

    def visit(node: Tree, parent: Optional[Tree] = None) -> Optional[DBoundary]:
        if is_leaf(node):
            return shift_word(label_of(node), parent, preterminal=None)

        if is_preterminal(node):
            boundary = shift_word(terminal_word(node), parent, preterminal=node)
            oracle.append(("NT", label_of(node)))
            oracle.append(("U-REDUCE", None))
            return boundary

        child_boundaries = [visit(child, parent=node) for child in children_of(node)]

        oracle.append(("NT", label_of(node)))
        active_boundaries = [boundary for boundary in child_boundaries if boundary is not None]

        if not active_boundaries:
            oracle.append(("REDUCE", None))
            return None

        if len(active_boundaries) > 1:
            raise ValueError(
                f"Multiple active delimiters reached node {label_of(node)}: {active_boundaries}"
            )

        active = active_boundaries[0]
        if active.target_id == id(node):
            oracle.append(("REDUCE", None))
            return None

        oracle.append(("D-REDUCE", None))
        return active

    final_boundary = visit(tree)
    if final_boundary is not None:
        raise ValueError(f"Unconsumed delimiter boundary: {final_boundary}")

    oracle.append(("FINISH", None))
    return oracle


## Oracle application

The applicator reconstructs a Stanza `Tree` from the sentence and oracle. Internally, the stack contains completed `Tree` objects and label markers. Dummy markers represent delimiter boundaries and are never included in the final tree.

In [4]:
@dataclass(frozen=True)
class LabelMarker:
    label: str
    dummy: bool = False

    def __repr__(self) -> str:
        return "NT()" if self.dummy else f"NT({self.label})"


StackItem = Union[Tree, LabelMarker]


def _is_dummy(item: StackItem) -> bool:
    return isinstance(item, LabelMarker) and item.dummy


def _is_real_marker(item: StackItem) -> bool:
    return isinstance(item, LabelMarker) and not item.dummy


def _item_to_string(item: StackItem) -> str:
    if isinstance(item, LabelMarker):
        return repr(item)
    return tree_to_ptb(item)


def apply_oracle_bottomup(
    sentence: Sequence[str],
    oracle: Sequence[OracleOp],
    *,
    verbose: bool = False,
) -> Tree:
    """Apply a delimiter-guided bottom-up oracle and return the reconstructed Stanza Tree."""

    stack: List[StackItem] = []
    word_index = 0
    finished = False
    final_tree: Optional[Tree] = None

    def stack_to_string() -> str:
        return "[ " + " | ".join(_item_to_string(item) for item in stack) + " ]" if stack else "[]"

    def show(step: int, action: str, value: Optional[str], when: str) -> None:
        if not verbose:
            return
        op = action if value is None else f"{action}({value})"
        remaining = " ".join(sentence[word_index:])
        print(f"{step:03d} {when:<6} {op:<18} stack={stack_to_string()} buffer=[ {remaining} ]")

    def shift(action_value: Optional[str], *, with_delimiter: bool) -> None:
        nonlocal word_index

        if word_index >= len(sentence):
            raise ValueError("SHIFT exceeds sentence length.")

        word = sentence[word_index]
        if action_value is not None and action_value != word:
            raise ValueError(f"SHIFT mismatch: oracle has {action_value!r}, sentence has {word!r}.")

        if with_delimiter:
            stack.append(LabelMarker("", dummy=True))
        stack.append(make_tree(word, []))
        word_index += 1

    def unary_reduce() -> None:
        if len(stack) < 2 or not _is_real_marker(stack[-1]) or not isinstance(stack[-2], Tree):
            raise ValueError(f"Invalid U-REDUCE configuration: {stack_to_string()}")

        marker = stack.pop()
        child = stack.pop()
        assert isinstance(marker, LabelMarker)
        assert isinstance(child, Tree)

        if not is_leaf(child):
            raise ValueError(f"U-REDUCE only applies to terminals, found: {tree_to_ptb(child)}")

        stack.append(make_tree(marker.label, [child]))

    def reduce_phrase(*, keep_delimiter: bool) -> None:
        if not stack or not _is_real_marker(stack[-1]):
            raise ValueError(f"REDUCE requires NT(label) on top: {stack_to_string()}")

        marker = stack[-1]
        assert isinstance(marker, LabelMarker)
        label_index = len(stack) - 1

        delimiter_index: Optional[int] = None
        for index in range(label_index - 1, -1, -1):
            if _is_dummy(stack[index]):
                delimiter_index = index
                break

        start = 0 if delimiter_index is None else delimiter_index + 1
        prefix = [] if delimiter_index is None else stack[:delimiter_index]
        children = stack[start:label_index]

        if not children or not all(isinstance(child, Tree) for child in children):
            raise ValueError(f"Invalid children for REDUCE: {stack_to_string()}")

        new_tree = make_tree(marker.label, children)  # type: ignore[arg-type]
        stack[:] = list(prefix)
        if keep_delimiter:
            stack.append(LabelMarker("", dummy=True))
        stack.append(new_tree)

    def finish() -> Tree:
        if word_index != len(sentence):
            raise ValueError(f"FINISH before consuming the sentence: {word_index}/{len(sentence)}")

        if len(stack) != 1 or not isinstance(stack[0], Tree):
            raise ValueError(f"FINISH requires exactly one completed tree: {stack_to_string()}")

        return stack[0]

    for step, (action, value) in enumerate(oracle, start=1):
        if finished:
            raise ValueError(f"Action after FINISH at step {step}: {action}")

        show(step, action, value, "before")

        if action == "SHIFT":
            shift(value, with_delimiter=False)
        elif action == "D-SHIFT":
            shift(value, with_delimiter=True)
        elif action == "NT":
            if value is None:
                raise ValueError("NT requires a label.")
            stack.append(LabelMarker(value))
        elif action == "U-REDUCE":
            unary_reduce()
        elif action == "REDUCE":
            reduce_phrase(keep_delimiter=False)
        elif action == "D-REDUCE":
            reduce_phrase(keep_delimiter=True)
        elif action == "FINISH":
            final_tree = finish()
            finished = True
        else:
            raise ValueError(f"Unknown action: {action}")

        show(step, action, value, "after")

    if not finished or final_tree is None:
        raise ValueError("Oracle ended without FINISH.")

    return final_tree


## End-to-end helpers

`run_pipeline` is the main function used in the examples. It returns the sentence, oracle, reconstructed tree, and exact-match result.

In [5]:
def format_oracle(oracle: Sequence[OracleOp]) -> str:
    """Format an oracle as one action per line."""
    lines = []
    for i, (action, value) in enumerate(oracle, start=1):
        op = action if value is None else f"{action}({value})"
        lines.append(f"{i:03d}: {op}")
    return "\n".join(lines)


def action_counts(oracle: Sequence[OracleOp]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for action, _ in oracle:
        counts[action] = counts.get(action, 0) + 1
    return counts


def run_pipeline(ptb_string: str, *, verbose: bool = False) -> dict[str, Any]:
    gold_tree = parse_ptb(ptb_string)
    sentence = get_sentence(gold_tree)
    oracle = get_oracle_bottomup(gold_tree)
    reconstructed_tree = apply_oracle_bottomup(sentence, oracle, verbose=verbose)

    return {
        "sentence": sentence,
        "oracle": oracle,
        "gold_tree": gold_tree,
        "reconstructed_tree": reconstructed_tree,
        "exact_match": same_tree(gold_tree, reconstructed_tree),
        "action_counts": action_counts(oracle),
    }

## Minimal example

In [6]:
ptb_string = "(TOP (S (NP (NNP Ms.) (NNP Haag)) (VP (VBZ plays) (NP (NNP Elianti))) (. .)))"

result = run_pipeline(ptb_string)

print("Sentence:", result["sentence"])
print("Action counts:", result["action_counts"])
print("Exact reconstruction:", result["exact_match"])
print()
print(format_oracle(result["oracle"]))
print()
print("Reconstructed tree:")
print(tree_to_ptb(result["reconstructed_tree"]))

assert result["exact_match"]

Sentence: ['Ms.', 'Haag', 'plays', 'Elianti', '.']
Action counts: {'D-SHIFT': 3, 'NT': 10, 'U-REDUCE': 5, 'SHIFT': 2, 'D-REDUCE': 2, 'REDUCE': 3, 'FINISH': 1}
Exact reconstruction: True

001: D-SHIFT(Ms.)
002: NT(NNP)
003: U-REDUCE
004: SHIFT(Haag)
005: NT(NNP)
006: U-REDUCE
007: NT(NP)
008: D-REDUCE
009: D-SHIFT(plays)
010: NT(VBZ)
011: U-REDUCE
012: D-SHIFT(Elianti)
013: NT(NNP)
014: U-REDUCE
015: NT(NP)
016: REDUCE
017: NT(VP)
018: REDUCE
019: SHIFT(.)
020: NT(.)
021: U-REDUCE
022: NT(S)
023: D-REDUCE
024: NT(TOP)
025: REDUCE
026: FINISH

Reconstructed tree:
(TOP (S (NP (NNP Ms.) (NNP Haag)) (VP (VBZ plays) (NP (NNP Elianti))) (. .)))


## Regression examples

These are the example trees used while developing the notebook. The assertion checks exact tree reconstruction, including punctuation.

In [7]:
examples = [
    "(TOP (S (NP (NNP Ms.) (NNP Haag)) (VP (VBZ plays) (NP (NNP Elianti))) (. .)))",
    "(TOP (S (NP (PRP We)) (VP (VBP 're) (VP (IN about) (S (VP (TO to) (VP (VB see) (SBAR (IN if) (S (NP (NN advertising)) (VP (VBZ works))))))))) (. .)))",
    "(TOP (S (NP (NP (JJ Influential) (NNS members)) (PP (IN of) (NP (DT the) (NNP House) (NNP Ways) (CC and) (NNP Means) (NNP Committee)))) (VP (VBD introduced) (NP (NP (NN legislation)) (SBAR (WHNP (WDT that)) (S (VP (MD would) (VP (VB restrict) (SBAR (WHADVP (WRB how)) (S (NP (DT the) (JJ new) (NN savings-and-loan) (NN bailout) (NN agency)) (VP (MD can) (VP (VB raise) (NP (NN capital)))))) (, ,) (S (VP (VBG creating) (NP (NP (DT another) (JJ potential) (NN obstacle)) (PP (TO to) (NP (NP (NP (DT the) (NN government) (POS 's)) (NN sale)) (PP (IN of) (NP (JJ sick) (NNS thrifts)))))))))))))) (. .)))",
]

for index, example in enumerate(examples, start=1):
    result = run_pipeline(example)
    print(
        f"example {index}: tokens={len(result['sentence'])}, "
        f"actions={len(result['oracle'])}, exact_match={result['exact_match']}, "
        f"counts={result['action_counts']}"
    )
    assert result["exact_match"]

example 1: tokens=5, actions=26, exact_match=True, counts={'D-SHIFT': 3, 'NT': 10, 'U-REDUCE': 5, 'SHIFT': 2, 'D-REDUCE': 2, 'REDUCE': 3, 'FINISH': 1}
example 2: tokens=9, actions=52, exact_match=True, counts={'D-SHIFT': 8, 'NT': 21, 'U-REDUCE': 9, 'D-REDUCE': 4, 'REDUCE': 8, 'SHIFT': 1, 'FINISH': 1}
example 3: tokens=37, actions=174, exact_match=True, counts={'D-SHIFT': 19, 'NT': 68, 'U-REDUCE': 37, 'SHIFT': 18, 'D-REDUCE': 12, 'REDUCE': 19, 'FINISH': 1}


## Optional verbose trace

Set `verbose=True` to inspect the stack and buffer before and after every transition.

In [8]:
# _ = run_pipeline(ptb_string, verbose=True)